# Multi-Agent RAG

## Vectordb setup

In [1]:
import chromadb
from chromadb.config import Settings

# Initialize ChromaDB with persistence
client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=Settings(
        allow_reset=True,
        anonymized_telemetry=False
    )
)

## Document Processing Pipeline

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

class DocumentProcessor:
    def __init__(self):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            separators=["\n\n", "\n", ".", " "]
        )
    
    def process_documents(self, documents):
        chunks = self.text_splitter.split_documents(documents)
        return chunks
        

## Save Vectordb

In [3]:
from langchain_community.document_loaders import PyPDFLoader
document_path = "/home/quang/Downloads/Kinh_te_cong_nghiep.pdf"
loader = PyPDFLoader(document_path)
documents = loader.load()
print(documents)

[Document(metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-07-12T23:23:53+16:23', 'author': 'LeHanh', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-07-12T23:23:53+16:23', 'sourcemodified': "D:20250712232353+16'23'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '/home/quang/Downloads/Kinh_te_cong_nghiep.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1'}, page_content='1\nĐẠI HỌC THÁI NGUYÊN\nTRƯỜNG ĐẠI HỌC KỸ THUẬT CÔNG\nNGHIỆP\nCHƯƠNG TRÌNH ĐÀO TẠO TỪ\nXA TRÌNH ĐỘ ĐẠI HỌC\nNGÀNH: KINH TẾ CÔNG NGHIỆP\nCHUYÊN NGÀNH: KẾ TOÁN DOANH\nNGHIỆP CÔNG NGHIỆP'), Document(metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-07-12T23:23:53+16:23', 'author': 'LeHanh', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-07-12T23:23:53+16:23', 'sourcemodified': "D:20250712232353+16'23'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '/home/quang/Downloads/Kinh_te_cong_nghiep.pdf', 'total_pages': 53,

In [4]:
raw_texts = [doc.page_content for doc in documents]

chunks = DocumentProcessor().process_documents(documents)

for chunk in chunks:
    chunk.page_content = chunk.page_content.replace("\n", " ")
    chunk.page_content = " ".join(chunk.page_content.split())

In [5]:
print(f"Number of chunks: {chunks[0].page_content}")

Number of chunks: 1 ĐẠI HỌC THÁI NGUYÊN TRƯỜNG ĐẠI HỌC KỸ THUẬT CÔNG NGHIỆP CHƯƠNG TRÌNH ĐÀO TẠO TỪ XA TRÌNH ĐỘ ĐẠI HỌC NGÀNH: KINH TẾ CÔNG NGHIỆP CHUYÊN NGÀNH: KẾ TOÁN DOANH NGHIỆP CÔNG NGHIỆP


In [6]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from chromadb.config import Settings
import chromadb

# Tạo thư mục nếu chưa tồn tại
persist_dir = "./chroma_db"
os.makedirs(persist_dir, exist_ok=True)

# Khởi tạo client với thư mục này
client = chromadb.Client(Settings(persist_directory=persist_dir))

# Lấy hoặc tạo collection
collection = client.get_or_create_collection(name="MuiltiAgentRAGCollection")

# Thêm dữ liệu
texts = [chunk.page_content for chunk in chunks]
metadatas = [{"source": chunk.metadata.get("source", "unknown")} for chunk in chunks]
embeddings = HuggingFaceEmbeddings(model_name="AITeamVN/Vietnamese_Embedding")

collection.add(
    documents=texts,
    metadatas=metadatas,
    embeddings=embeddings.embed_documents(texts),
    ids=[str(i) for i in range(len(texts))]
)


/home/quang/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
from langchain.vectorstores import Chroma
# 1. Tạo embeddings (dùng khi query)
embeddings = HuggingFaceEmbeddings(model_name="AITeamVN/Vietnamese_Embedding", model_kwargs={"device": "cpu"})

# 2. Kết nối ChromaDB
chroma_db = Chroma(
    persist_directory="./db",  # thư mục bạn đã lưu DB
    embedding_function=embeddings
)

# 3. Tạo retriever
retriever = chroma_db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "maximal_marginal_relevance": True,
        "diversity_penalty": 0.3
    }
)


## Building Multi-Agent RAG System

### Build tool

In [15]:
from crewai.tools import BaseTool
import google.generativeai as genai
from dotenv import load_dotenv
import os

load_dotenv()

class GeminiGoogleSearchTool(BaseTool):
    name:str = "GeminiGoogleSearch"
    description:str = "Search the web using Google's native Gemini Search grounding."

    def _run(self, query: str) -> str:
        # Cấu hình API key
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
        
        # Khởi tạo model
        model = genai.GenerativeModel('gemini-2.5-flash')
        
        # Tạo prompt bao gồm yêu cầu tìm kiếm
        search_prompt = f"Please search and provide information about: {query}"
        
        # Gọi model để lấy kết quả
        response = model.generate_content(search_prompt)
        
        return response.text

# Now modify your RAGTool implementation
class RAGTool(BaseTool):
    name: str = "RAGTool"
    description: str = "A tool that combines ChromaDB retrieval with web search capabilities"
    
    retriever: any
    web_search_tool: BaseTool

    def _run(self, query: str) -> str:
        try:
            docs = self.retriever.get_relevant_documents(query)
            
            if docs:
                processed_docs = []
                for doc in docs:
                    source = doc.metadata.get('source', 'Unknown source')
                    content = doc.page_content
                    processed_docs.append(f"Source: {source}\nContent: {content}")
                
                context = "\n\n".join(processed_docs)
                return f"From local knowledge base:\n{context}"
            else:
                print("No relevant documents found in local DB, trying web search...")
                web_results = self.web_search_tool._run(query)
                return f"From web search:\n{web_results}"
                
        except Exception as e:
            print(f"Error during retrieval: {str(e)}")
            try:
                web_results = self.web_search_tool._run(query)
                return f"From web search (after local DB error):\n{web_results}"
            except Exception as web_e:
                return f"Failed to retrieve information: {str(web_e)}"

# Initialize tools
gemini_search_tool = GeminiGoogleSearchTool()
rag_tool = RAGTool(retriever=retriever, web_search_tool=gemini_search_tool)

/home/quang/miniconda3/envs/paraline/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:628: UserWarning: <built-in function any> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


### Test tool GoogleSearch 

In [16]:
tool = GeminiGoogleSearchTool()
query = "Who won the Euro 2024?"
result = tool.run(query)
print("Query:", query)
print("Result:", result)

Using Tool: GeminiGoogleSearch
Query: Who won the Euro 2024?
Result: The UEFA Euro 2024 tournament is currently underway!

**The final match is scheduled for July 14, 2024.** Therefore, the winner has not yet been determined.

The tournament began on June 14, 2024, and will conclude on July 14, 2024, in Germany.

Please check back after July 14th for information on the winner!


### Test tool RAG

In [17]:
# Test retrieval
query = "Mục tiêu chung của kinh tế công nghiệp là gì?"
result = rag_tool.run(query)
print(result)

Using Tool: RAGTool


/tmp/ipykernel_17302/1575713056.py:37: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = self.retriever.get_relevant_documents(query)


Error during retrieval: Collection.query() got an unexpected keyword argument 'fetch_k'
From web search (after local DB error):
Kinh tế công nghiệp (Industrial Economics) là một phân ngành của kinh tế học tập trung vào nghiên cứu cấu trúc của các ngành công nghiệp, hành vi của các doanh nghiệp và tác động của các yếu tố này đến thị trường và phúc lợi xã hội.

**Mục tiêu chung của kinh tế công nghiệp** có thể được tóm tắt như sau:

1.  **Phân tích và Hiểu rõ hành vi doanh nghiệp và cấu trúc thị trường:**
    *   Nghiên cứu cách các doanh nghiệp đưa ra quyết định về giá cả, sản lượng, đầu tư, đổi mới và chiến lược cạnh tranh.
    *   Phân tích các dạng cấu trúc thị trường khác nhau (độc quyền, độc quyền nhóm, cạnh tranh hoàn hảo, cạnh tranh độc quyền) và tác động của chúng đến hành vi của doanh nghiệp và kết quả thị trường.
    *   Hiểu rõ các yếu tố định hình cấu trúc thị trường như rào cản gia nhập, quy mô kinh tế, công nghệ.

2.  **Đánh giá Hiệu quả kinh tế:**
    *   **Hiệu quả phân 

### Agent Implementation

In [18]:
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool, WebsiteSearchTool
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
load_dotenv()

# Kiểm tra OPENAI_API_KEY
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Vui lòng cung cấp OPENAI_API_KEY trong file .env")

# Initialize the language model using OpenAI
llm = ChatOpenAI(
    model="gpt-4o-mini",  # Hoặc "gpt-4" nếu bạn cần model mạnh hơn
    temperature=0.7,
    verbose=True
)

# Research Agent
research_agent = Agent(
    role="Research Specialist",
    goal="Retrieve and analyze relevant documents from both local knowledge base and web sources",
    backstory="You are an expert at finding and analyzing relevant information from multiple sources. You first check the local knowledge base for relevant information, and if needed, expand your search to web sources.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[rag_tool]  # Using the combined RAG tool
)

### Task Definition

In [19]:
# Research Task
research_task = Task(
    description="Research and answer the following query: {topic}",
    agent=research_agent,
    expected_output="A direct and accurate answer to the query based on available information."
)

### Crew Assembly

In [20]:
# Create the crew
analysis_crew = Crew(
    agents=[research_agent],
    tasks=[research_task],
    verbose=True,
    process="sequential"  # or "hierarchical" for complex scenarios
)

# Execute the workflow
def run_analysis(query):
    result = analysis_crew.kickoff(inputs={"topic": query})
    return result

### Execution

In [21]:
query = "Who won the Euro 2024?"
result = run_analysis(query)
print("Query:", query)
print("Final Result:", result)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 84855b1f-5d34-4e63-a78a-ae0665becfb2                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Task: Research and answer the following query: Who won the Euro 2024?                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/home/quang/miniconda3/envs/paraline/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/home/quang/miniconda3/envs/paraline/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Error during retrieval: Collection.query() got an unexpected keyword argument 'fetch_k'

...

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  From web search (after local DB error):                                                                        │
│  The UEFA Euro 2024 tournament has not concluded yet, so there isn't a winner to announce!                      │
│                                                                                                                 │
│  The tournament is scheduled to take place from **June 14 to July 14, 2024**, in Germany.                       │
│                                                                                                                 │
│  You'll need to wait until after the final match on July 14th to find out who lifts the trophy!                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The UEFA Euro 2024 tournament has not concluded yet, so there isn't a winner to announce! The tournament is    │
│  scheduled to take place from June 14 to July 14, 2024, in Germany. You'll need to wait until after the final   │
│  match on July 14th to find out who lifts the trophy!                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: a352fdb3-9bbf-4d3f-9f4a-f8f5afc4eade                                                                     │
│  Agent: Research Specialist                                                                                     │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 84855b1f-5d34-4e63-a78a-ae0665becfb2                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: The UEFA Euro 2024 tournament has not concluded yet, so there isn't a winner to announce! The    │
│  tournament is scheduled to take place from June 14 to July 14, 2024, in Germany. You'll need to wait until     │
│  after the final match on July 14th to find out who lifts the trophy!                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Query: Who won the Euro 2024?
Final Result: The UEFA Euro 2024 tournament has not concluded yet, so there isn't a winner to announce! The tournament is scheduled to take place from June 14 to July 14, 2024, in Germany. You'll need to wait until after the final match on July 14th to find out who lifts the trophy!
